In [20]:
import math
import re
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

try:
    import pulp
except Exception:
    pulp = None

EPS = 1e-12


# TRIANGULAR FUZZY SCALE

In [21]:
LINGUISTIC_SCALE = {
    "AL": (1.00, 1.50, 2.50),
    "VL": (1.50, 2.50, 3.50),
    "L":  (2.50, 3.50, 4.50),
    "ML": (3.50, 4.50, 5.50),
    "E":  (4.50, 5.50, 6.50),
    "MH": (5.50, 6.50, 7.50),
    "H":  (6.50, 7.50, 8.50),
    "VH": (7.50, 8.50, 9.50),
    "AH": (8.50, 9.00, 10.00),
}
LINGUISTIC_OPTIONS = list(LINGUISTIC_SCALE.keys())


# HELPERS

In [22]:
def parse_number(x):
    """Convert strings like '7,400,000.00' to float; keep numerics as-is."""
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return np.nan
    if isinstance(x, (int, float, np.number)):
        return float(x)
    s = str(x).strip()
    if s == "":
        return np.nan
    s = s.replace(",", "")
    if re.match(r"^\(\s*[-+]?\d+(\.\d+)?\s*\)$", s):
        s = "-" + s.strip("()").strip()
    return float(s)

def safe_normalize_to_1(v: np.ndarray) -> np.ndarray:
    v = np.asarray(v, dtype=float)
    s = np.nansum(v)
    if len(v) == 0:
        raise ValueError("Vector length cannot be zero.")
    if s <= 0 or np.isclose(s, 0.0) or np.isnan(s):
        return np.ones_like(v) / len(v)
    return v / s

def ensure_2d(a):
    a = np.asarray(a, dtype=float)
    if a.ndim == 1:
        return a.reshape(1, -1)
    return a

def safe_pos(x: float, eps: float = EPS) -> float:
    return max(float(x), eps)

def defuzz_tfn(tfn: Tuple[float, float, float]) -> float:
    return (tfn[0] + 4 * tfn[1] + tfn[2]) / 6.0

def to_bc_label(x: str) -> str:
    s = str(x).strip().upper()
    if s in {"B", "BENEFIT", "MAX"}:
        return "B"
    return "C"

def validate_linguistic_df(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for c in out.columns:
        out[c] = out[c].astype(str).str.strip().str.upper()
        out[c] = out[c].where(out[c].isin(LINGUISTIC_OPTIONS), "E")
    return out

def check_missing_df(df: pd.DataFrame, name: str):
    if df.isna().any().any():
        raise ValueError(f"{name} contains blank or invalid cells.")

def check_missing_array(arr, name: str):
    arr = np.asarray(arr, dtype=float)
    if np.isnan(arr).any():
        raise ValueError(f"{name} contains blank or invalid values.")
    return arr

def validate_expert_weights(linguistic_df: pd.DataFrame, expert_weights: List[float]) -> np.ndarray:
    expert_weights = np.asarray(expert_weights, dtype=float).ravel()
    n_experts = linguistic_df.shape[1]

    if expert_weights.size != n_experts:
        raise ValueError(
            f"Number of expert weights ({expert_weights.size}) must equal the number of expert columns "
            f"in linguistic_df ({n_experts})."
        )

    if np.any(expert_weights < 0):
        raise ValueError("Expert weights must be non-negative.")

    if not np.isclose(expert_weights.sum(), 1.0, atol=1e-8):
        raise ValueError(
            f"Expert weights must sum to 1.0. Current sum = {expert_weights.sum():.10f}"
        )

    return expert_weights

# PHASE 1: OBJECTIVE WEIGHTS (m-ITARA + EWCS)

In [23]:
def mitara_stage1_independent_weights(D, AL, IT):
    D = ensure_2d(D)
    AL = np.asarray(AL, dtype=float).reshape(1, -1)
    IT = np.asarray(IT, dtype=float).ravel()

    m, n = D.shape
    if AL.shape[1] != n or IT.shape[0] != n:
        raise ValueError("Dimension mismatch in Stage I inputs.")

    D_ext = np.vstack([D, AL])
    col_sum = np.sum(D_ext, axis=0)
    if np.any(np.isclose(col_sum, 0.0)):
        raise ValueError("Some criterion column sums are 0, so normalization cannot be performed.")

    A = D_ext / col_sum
    NIT = IT / col_sum

    beta = np.sort(A, axis=0)
    gamma = beta[1:, :] - beta[:-1, :]
    delta = np.maximum(gamma - NIT.reshape(1, -1), 0.0)
    nu = np.sqrt(np.sum(delta ** 2, axis=0))
    w1 = safe_normalize_to_1(nu)

    return {
        "D_ext": D_ext,
        "A": A,
        "NIT": NIT,
        "beta": beta,
        "gamma": gamma,
        "delta": delta,
        "nu": nu,
        "w1": w1,
    }

def stage2_dependent_weights_corr(D, types_bc):
    D = ensure_2d(D)
    types_bc = [str(t).strip().upper() for t in types_bc]

    m, n = D.shape
    if len(types_bc) != n:
        raise ValueError("types_bc length must match the number of criteria.")

    X = np.zeros_like(D, dtype=float)

    for j in range(n):
        col = D[:, j].astype(float)
        mn, mx = np.nanmin(col), np.nanmax(col)
        if np.isclose(mx - mn, 0.0) or np.isnan(mx - mn):
            X[:, j] = 0.0
            continue

        if types_bc[j] == "B":
            X[:, j] = (col - mn) / (mx - mn)
        elif types_bc[j] == "C":
            X[:, j] = (mx - col) / (mx - mn)
        else:
            raise ValueError(f"Invalid criterion type at column {j+1}: {types_bc[j]}. Use B or C.")

    R = np.corrcoef(X, rowvar=False)
    R = np.nan_to_num(R, nan=0.0)

    phi = np.sum(1.0 - R, axis=1)
    w2 = safe_normalize_to_1(phi)

    return {"X": X, "R": R, "phi": phi, "w2": w2}

def ewcs_fusion(w1, w2):
    w1 = np.asarray(w1, dtype=float).ravel()
    w2 = np.asarray(w2, dtype=float).ravel()

    if w1.shape != w2.shape:
        raise ValueError("w1 and w2 must have the same length.")

    a11 = float(np.dot(w1, w1))
    a12 = float(np.dot(w1, w2))
    a22 = float(np.dot(w2, w2))

    A = np.array([[a11, a12], [a12, a22]], dtype=float)
    b = np.array([a11, a22], dtype=float)

    det = np.linalg.det(A)
    if np.isclose(det, 0.0):
        p_raw = np.array([1.0, 1.0])
    else:
        p_raw = np.linalg.solve(A, b)

    p_star = np.abs(p_raw)
    p_star = p_star / np.sum(p_star)

    wagg = p_star[0] * w1 + p_star[1] * w2
    wagg = safe_normalize_to_1(wagg)

    return {"p_raw": p_raw, "p_star": p_star, "wagg": wagg}


# PHASE 2: SUBJECTIVE WEIGHTS (Trig Fuzzy OPA)

In [24]:
def trig_geom_component(values, weights):
    if len(values) != len(weights):
        raise ValueError(
            f"Length mismatch in trigonometric aggregation: {len(values)} values but {len(weights)} weights."
        )

    s = sum(values)
    if s == 0:
        return 0.0

    prod = 1.0
    for v, w in zip(values, weights):
        ratio = v / s
        term = np.cos(np.pi * ratio / 2.0) ** w
        prod *= term

    prod = np.clip(prod, -1.0, 1.0)
    return s * (2.0 / np.pi) * np.arccos(prod)

def aggregate_tfn(tfn_list, weights):
    ls = [t[0] for t in tfn_list]
    ms = [t[1] for t in tfn_list]
    us = [t[2] for t in tfn_list]
    l = trig_geom_component(ls, weights)
    m = trig_geom_component(ms, weights)
    u = trig_geom_component(us, weights)
    return (l, m, u)

def solve_fuzzy_opa(coeff_list, n):
    if pulp is None:
        raise RuntimeError("PuLP is not installed. Install it first: pip install pulp")

    prob = pulp.LpProblem("Triangular_Fuzzy_OPA", pulp.LpMaximize)

    w_l = [pulp.LpVariable(f"w_l_{i}", lowBound=0) for i in range(n)]
    w_m = [pulp.LpVariable(f"w_m_{i}", lowBound=0) for i in range(n)]
    w_u = [pulp.LpVariable(f"w_u_{i}", lowBound=0) for i in range(n)]

    psi_l = pulp.LpVariable("psi_l", lowBound=0)
    psi_m = pulp.LpVariable("psi_m", lowBound=0)
    psi_u = pulp.LpVariable("psi_u", lowBound=0)

    prob += (psi_l + 2 * psi_m + psi_u) / 4.0

    for i in range(n):
        prob += w_l[i] <= w_m[i]
        prob += w_m[i] <= w_u[i]

    prob += pulp.lpSum(w_l) == 0.8
    prob += pulp.lpSum(w_m) == 1.0
    prob += pulp.lpSum(w_u) == 1.2

    for a in range(n - 1):
        prob += coeff_list[a][0] * (w_l[a] - w_u[a + 1]) >= psi_l
        prob += coeff_list[a][1] * (w_m[a] - w_m[a + 1]) >= psi_m
        prob += coeff_list[a][2] * (w_u[a] - w_l[a + 1]) >= psi_u

    prob += coeff_list[n - 1][0] * w_l[n - 1] >= psi_l
    prob += coeff_list[n - 1][1] * w_m[n - 1] >= psi_m
    prob += coeff_list[n - 1][2] * w_u[n - 1] >= psi_u

    status = prob.solve(pulp.PULP_CBC_CMD(msg=False))
    if status != 1:
        return None, None

    weights = []
    for i in range(n):
        weights.append((
            max(0, pulp.value(w_l[i])),
            max(0, pulp.value(w_m[i])),
            max(0, pulp.value(w_u[i])),
        ))

    psi = (
        max(0, pulp.value(psi_l)),
        max(0, pulp.value(psi_m)),
        max(0, pulp.value(psi_u)),
    )
    return weights, psi

def run_fuzzy_opa(criteria_names, linguistic_df, expert_weights):
    n = len(criteria_names)
    linguistic_df = validate_linguistic_df(linguistic_df)
    expert_weights = validate_expert_weights(linguistic_df, expert_weights)

    theta = []
    defuzz_values = []
    for crit in criteria_names:
        tfn_list = [LINGUISTIC_SCALE[linguistic_df.loc[crit, e]] for e in linguistic_df.columns]
        aggregated = aggregate_tfn(tfn_list, expert_weights)
        theta.append(aggregated)
        defuzz_values.append(defuzz_tfn(aggregated))

    positive_lowers = [t[0] for t in theta if t[0] > 0]
    min_l = min(positive_lowers) if positive_lowers else 1.0

    coeff = []
    for t in theta:
        if min(t) <= 0:
            coeff.append((0.0, 0.0, 0.0))
        else:
            coeff.append((min_l / t[2], min_l / t[1], min_l / t[0]))

    sorted_indices = np.argsort(defuzz_values)[::-1]
    coeff_sorted = [coeff[idx] for idx in sorted_indices]
    criteria_sorted = [criteria_names[idx] for idx in sorted_indices]

    weights_sorted, psi = solve_fuzzy_opa(coeff_sorted, n)
    if weights_sorted is None:
        raise ValueError("Fuzzy OPA optimization failed. Check linguistic data and expert weights.")

    weights = [None] * n
    for rank, idx in enumerate(sorted_indices):
        weights[idx] = weights_sorted[rank]

    theta_df = pd.DataFrame({
        "Criterion": criteria_names,
        "Theta_l": [t[0] for t in theta],
        "Theta_m": [t[1] for t in theta],
        "Theta_u": [t[2] for t in theta],
        "Defuzzified": defuzz_values,
    })

    coeff_df = pd.DataFrame({
        "Criterion": criteria_names,
        "Coeff_l": [c[0] for c in coeff],
        "Coeff_m": [c[1] for c in coeff],
        "Coeff_u": [c[2] for c in coeff],
    })

    weights_df = pd.DataFrame({
        "Criterion": criteria_names,
        "Weight_l": [w[0] for w in weights],
        "Weight_m": [w[1] for w in weights],
        "Weight_u": [w[2] for w in weights],
        "Defuzzified": [defuzz_tfn(w) for w in weights],
    })

    ranked_df = pd.DataFrame({
        "Rank": list(range(1, n + 1)),
        "Criterion": [criteria_names[idx] for idx in sorted_indices],
        "Weight_l": [weights_sorted[r][0] for r in range(n)],
        "Weight_m": [weights_sorted[r][1] for r in range(n)],
        "Weight_u": [weights_sorted[r][2] for r in range(n)],
        "Defuzzified": [defuzz_tfn(weights_sorted[r]) for r in range(n)],
    })

    return {
        "theta": theta,
        "defuzz_values": defuzz_values,
        "coeff": coeff,
        "sorted_indices": sorted_indices,
        "criteria_sorted": criteria_sorted,
        "weights": weights,
        "weights_sorted": weights_sorted,
        "psi": psi,
        "theta_df": theta_df,
        "coeff_df": coeff_df,
        "weights_df": weights_df,
        "ranked_df": ranked_df,
    }


# PHASE 3 PREP: FUZZY INPUT CONVERSION

In [25]:
def build_fuzzy_df_from_crisp_percent(raw_df: pd.DataFrame, epsilon: float = 0.10) -> pd.DataFrame:
    cols = []
    rows = []

    for c in raw_df.columns:
        cols.extend([f"{c}_l", f"{c}_m", f"{c}_u"])

    for _, r in raw_df.iterrows():
        row_vals = []
        for c in raw_df.columns:
            x = float(r[c])
            l = x * (1.0 - epsilon)
            m = x
            u = x * (1.0 + epsilon)
            trip = sorted([l, m, u])
            row_vals.extend(trip)
        rows.append(row_vals)

    return pd.DataFrame(rows, index=raw_df.index, columns=cols)

def sanitize_fuzzy_df(fuzzy_df: pd.DataFrame, criteria: List[str], alternatives: List[str]) -> pd.DataFrame:
    out = fuzzy_df.copy().astype(float)
    for c in criteria:
        trip = out[[f"{c}_l", f"{c}_m", f"{c}_u"]].values
        trip = np.sort(trip, axis=1)
        out[[f"{c}_l", f"{c}_m", f"{c}_u"]] = trip
    return out.reindex(index=alternatives)

def fuzzy_df_to_nested_matrix(fuzzy_df: pd.DataFrame, criteria: List[str], alternatives: List[str]):
    matrix = []
    for alt in alternatives:
        row = []
        for c in criteria:
            trip = (
                float(fuzzy_df.loc[alt, f"{c}_l"]),
                float(fuzzy_df.loc[alt, f"{c}_m"]),
                float(fuzzy_df.loc[alt, f"{c}_u"]),
            )
            row.append(tuple(sorted(trip)))
        matrix.append(row)
    return matrix


# PHASE 3: FUZZY BONFERRONI CoCoSo (REVISED: EXCEL-STYLE CRISP RANKING)

In [26]:
def normalize_cocoso_bonferroni(decision, types_bc):
    n_alt = len(decision)
    n_crit = len(types_bc)
    norm = [[(0.0, 0.0, 0.0) for _ in range(n_crit)] for _ in range(n_alt)]

    for j in range(n_crit):
        typ = to_bc_label(types_bc[j])

        if typ == "B":
            max_u = max(decision[i][j][2] for i in range(n_alt))
            max_u = safe_pos(max_u)
            for i in range(n_alt):
                l, m, u = decision[i][j]
                norm[i][j] = (l / max_u, m / max_u, u / max_u)
        else:
            min_l = min(decision[i][j][0] for i in range(n_alt))
            min_l = safe_pos(min_l)
            for i in range(n_alt):
                l, m, u = decision[i][j]
                l = safe_pos(l)
                m = safe_pos(m)
                u = safe_pos(u)
                norm[i][j] = (min_l / u, min_l / m, min_l / l)

    return norm

def compute_bonferroni(norm_matrix, weights, phi1=1.0, phi2=1.0):
    weights = safe_normalize_to_1(pd.Series(weights).astype(float).values)

    n_alt = len(norm_matrix)
    n_crit = len(weights)

    if n_crit < 2:
        raise ValueError("At least two criteria are required for fuzzy Bonferroni CoCoSo.")

    scob = []
    pcob = []
    exp_term = 1.0 / safe_pos(phi1 + phi2)

    for a in range(n_alt):
        s_l = 0.0
        s_m = 0.0
        s_u = 0.0

        log_p_l = 0.0
        log_p_m = 0.0
        log_p_u = 0.0

        for i in range(n_crit):
            wi = min(max(weights[i], EPS), 1.0 - EPS)
            denom = 1.0 - wi

            for j in range(n_crit):
                if i == j:
                    continue

                wj = weights[j]
                term = (wi * wj) / denom

                gi_l, gi_m, gi_u = norm_matrix[a][i]
                gj_l, gj_m, gj_u = norm_matrix[a][j]

                # SCoB
                s_l += term * (safe_pos(gi_l) ** phi1) * (safe_pos(gj_l) ** phi2)
                s_m += term * (safe_pos(gi_m) ** phi1) * (safe_pos(gj_m) ** phi2)
                s_u += term * (safe_pos(gi_u) ** phi1) * (safe_pos(gj_u) ** phi2)

                # PCoB
                base_l = safe_pos(phi1 * gi_l + phi2 * gj_l)
                base_m = safe_pos(phi1 * gi_m + phi2 * gj_m)
                base_u = safe_pos(phi1 * gi_u + phi2 * gj_u)

                log_p_l += term * math.log(base_l)
                log_p_m += term * math.log(base_m)
                log_p_u += term * math.log(base_u)

        s_l = safe_pos(s_l) ** exp_term
        s_m = safe_pos(s_m) ** exp_term
        s_u = safe_pos(s_u) ** exp_term
        scob.append((s_l, s_m, s_u))

        p_l = math.exp(log_p_l) / safe_pos(phi1 + phi2)
        p_m = math.exp(log_p_m) / safe_pos(phi1 + phi2)
        p_u = math.exp(log_p_u) / safe_pos(phi1 + phi2)
        pcob.append((p_l, p_m, p_u))

    return scob, pcob

def relative_significance(scob, pcob, pi=0.5):
    """
    Revised version:
    - First defuzzify SCoB and PCoB using the SAME TFN formula
      Crisp = (l + 4m + u) / 6
    - Then compute Kia, Kib, Kic from the crisp values
      (Excel-style ranking logic).
    """
    scob_crisp = np.array([defuzz_tfn(s) for s in scob], dtype=float)
    pcob_crisp = np.array([defuzz_tfn(p) for p in pcob], dtype=float)

    sum_scob = safe_pos(np.sum(scob_crisp))
    sum_pcob = safe_pos(np.sum(pcob_crisp))

    min_scob = safe_pos(np.min(scob_crisp))
    min_pcob = safe_pos(np.min(pcob_crisp))

    max_scob = safe_pos(np.max(scob_crisp))
    max_pcob = safe_pos(np.max(pcob_crisp))

    # Kia
    kia = (scob_crisp + pcob_crisp) / safe_pos(sum_scob + sum_pcob)

    # Kib
    kib = (scob_crisp / min_scob) + (pcob_crisp / min_pcob)

    # Kic
    kic = (pi * scob_crisp + (1.0 - pi) * pcob_crisp) / safe_pos(
        pi * max_scob + (1.0 - pi) * max_pcob
    )

    meta = {
        "sum_Crisp_SCoB": sum_scob,
        "sum_Crisp_PCoB": sum_pcob,
        "min_Crisp_SCoB": min_scob,
        "min_Crisp_PCoB": min_pcob,
        "max_Crisp_SCoB": max_scob,
        "max_Crisp_PCoB": max_pcob,
    }

    return scob_crisp, pcob_crisp, kia, kib, kic, meta

def final_scores_bonferroni(
    scob,
    pcob,
    scob_crisp,
    pcob_crisp,
    kia,
    kib,
    kic,
    alternative_names=None,
):
    n_alt = len(kia)
    if alternative_names is None:
        alternative_names = [f"A{i+1}" for i in range(n_alt)]

    rows = []
    for i in range(n_alt):
        K = (
            (safe_pos(kia[i]) * safe_pos(kib[i]) * safe_pos(kic[i])) ** (1.0 / 3.0)
            + (kia[i] + kib[i] + kic[i]) / 3.0
        )

        rows.append([
            alternative_names[i],

            scob[i][0], scob[i][1], scob[i][2],
            pcob[i][0], pcob[i][1], pcob[i][2],

            scob_crisp[i],
            pcob_crisp[i],

            kia[i],
            kib[i],
            kic[i],

            K,      # Excel-style final score
            K,      # keep "Crisp" for compatibility with your existing print statement
        ])

    df = pd.DataFrame(
        rows,
        columns=[
            "Alternative",
            "SCoB_l", "SCoB_m", "SCoB_u",
            "PCoB_l", "PCoB_m", "PCoB_u",
            "Crisp_SCoB", "Crisp_PCoB",
            "Kia", "Kib", "Kic",
            "K", "Crisp",
        ],
    )

    df["Rank"] = df["K"].rank(ascending=False, method="min").astype(int)
    return df.sort_values(["K", "Alternative"], ascending=[False, True]).reset_index(drop=True)

def cocoso_bonferroni_from_app(fuzzy_df, types_bc, final_weights, phi1=1.0, phi2=1.0, pi=0.5):
    criteria = [c[:-2] for c in fuzzy_df.columns if c.endswith("_l")]
    alternatives = fuzzy_df.index.astype(str).tolist()

    decision = fuzzy_df_to_nested_matrix(fuzzy_df, criteria, alternatives)
    norm_matrix = normalize_cocoso_bonferroni(decision, types_bc)
    weights = pd.Series(final_weights, index=criteria).astype(float)

    scob, pcob = compute_bonferroni(norm_matrix, weights, phi1=phi1, phi2=phi2)

    scob_crisp, pcob_crisp, kia, kib, kic, crisp_meta = relative_significance(
        scob, pcob, pi=pi
    )

    ranking_df = final_scores_bonferroni(
        scob=scob,
        pcob=pcob,
        scob_crisp=scob_crisp,
        pcob_crisp=pcob_crisp,
        kia=kia,
        kib=kib,
        kic=kic,
        alternative_names=alternatives,
    )

    norm_rows = []
    for i, alt in enumerate(alternatives):
        row = {"Alternative": alt}
        for j, c in enumerate(criteria):
            row[f"{c}_l"] = norm_matrix[i][j][0]
            row[f"{c}_m"] = norm_matrix[i][j][1]
            row[f"{c}_u"] = norm_matrix[i][j][2]
        norm_rows.append(row)
    norm_df = pd.DataFrame(norm_rows).set_index("Alternative")

    scob_df = pd.DataFrame(
        scob,
        columns=["SCoB_l", "SCoB_m", "SCoB_u"],
        index=alternatives,
    )
    scob_df["Crisp_SCoB"] = scob_crisp

    pcob_df = pd.DataFrame(
        pcob,
        columns=["PCoB_l", "PCoB_m", "PCoB_u"],
        index=alternatives,
    )
    pcob_df["Crisp_PCoB"] = pcob_crisp

    psi_a_df = pd.DataFrame({"Kia": kia}, index=alternatives)
    psi_b_df = pd.DataFrame({"Kib": kib}, index=alternatives)
    psi_c_df = pd.DataFrame({"Kic": kic}, index=alternatives)

    cocoso_meta = {
        "phi1": phi1,
        "phi2": phi2,
        "pi": pi,
        **crisp_meta,
    }

    return ranking_df, cocoso_meta, norm_df, scob_df, pcob_df, psi_a_df, psi_b_df, psi_c_df


# INTEGRATED PIPELINE

In [27]:
def run_integrated_analysis(
    D_df: pd.DataFrame,
    types: List[str],
    AL,
    IT,
    linguistic_df: pd.DataFrame,
    expert_weights: List[float],
    fuzzy_df: Optional[pd.DataFrame] = None,
    epsilon: float = 0.10,
    phi1: float = 1.0,
    phi2: float = 1.0,
    pi_coef: float = 0.5,
    WL=None,
) -> Dict[str, pd.DataFrame]:
    D_df = D_df.copy().apply(lambda col: col.map(parse_number))
    check_missing_df(D_df, "Decision matrix")

    criteria = D_df.columns.astype(str).tolist()
    alternatives = D_df.index.astype(str).tolist()

    types = [to_bc_label(t) for t in types]
    if len(types) != len(criteria):
        raise ValueError("Length of types must match the number of criteria.")

    AL = check_missing_array(AL, "AL")
    IT = check_missing_array(IT, "IT")

    linguistic_df = validate_linguistic_df(linguistic_df.copy())
    if list(linguistic_df.index.astype(str)) != criteria:
        linguistic_df = linguistic_df.reindex(criteria)

    expert_weights = validate_expert_weights(linguistic_df, expert_weights)

    D_num = D_df.to_numpy(dtype=float)
    sigma = D_num.std(axis=0, ddof=0)
    it_ok = IT < sigma

    stage1 = mitara_stage1_independent_weights(D_num, AL, IT)
    stage2 = stage2_dependent_weights_corr(D_num, types)
    stage3_obj = ewcs_fusion(stage1["w1"], stage2["w2"])

    objective_summary = pd.DataFrame({
        "Criterion": criteria,
        "Type": types,
        "sigma": sigma,
        "IT": IT,
        "IT<sigma": it_ok,
        "w1_independent": stage1["w1"],
        "w2_dependent": stage2["w2"],
        "objective_weight": stage3_obj["wagg"],
    })
    objective_summary["Objective_Rank"] = objective_summary["objective_weight"].rank(ascending=False, method="dense").astype(int)

    opa_result = run_fuzzy_opa(criteria, linguistic_df, expert_weights.tolist())
    subjective_crisp = opa_result["weights_df"]["Defuzzified"].to_numpy(dtype=float)

    final_stage = ewcs_fusion(stage3_obj["wagg"], subjective_crisp)
    final_weights = pd.Series(final_stage["wagg"], index=criteria, name="final_weight")

    final_weight_df = pd.DataFrame({
        "Criterion": criteria,
        "Objective_weight": stage3_obj["wagg"],
        "Subjective_weight": subjective_crisp,
        "Final_weight": final_weights.values,
    })
    final_weight_df["Rank"] = final_weight_df["Final_weight"].rank(ascending=False, method="min").astype(int)
    final_weight_df = final_weight_df.sort_values("Final_weight", ascending=False).reset_index(drop=True)

    if fuzzy_df is None:
        fuzzy_df = build_fuzzy_df_from_crisp_percent(D_df.astype(float), epsilon=epsilon)
    else:
        fuzzy_df = sanitize_fuzzy_df(fuzzy_df, criteria=criteria, alternatives=alternatives)

    ranking_df, cocoso_meta, norm_df, scob_df, pcob_df, psi_a_df, psi_b_df, psi_c_df = cocoso_bonferroni_from_app(
        fuzzy_df=fuzzy_df,
        types_bc=types,
        final_weights=final_weights.values,
        phi1=float(phi1),
        phi2=float(phi2),
        pi=float(pi_coef),
    )

    return {
        "DecisionMatrix": D_df,
        "ObjectiveSummary": objective_summary,
        "StageI_A": pd.DataFrame(stage1["A"], index=alternatives + ["AL"], columns=criteria),
        "StageI_beta": pd.DataFrame(stage1["beta"], columns=criteria),
        "StageI_gamma": pd.DataFrame(stage1["gamma"], columns=criteria),
        "StageI_delta": pd.DataFrame(stage1["delta"], columns=criteria),
        "StageI_NIT": pd.DataFrame({"NIT": stage1["NIT"]}, index=criteria),
        "StageI_nu": pd.DataFrame({"nu": stage1["nu"]}, index=criteria),
        "StageII_X": pd.DataFrame(stage2["X"], index=alternatives, columns=criteria),
        "StageII_R": pd.DataFrame(stage2["R"], index=criteria, columns=criteria),
        "StageII_phi_w2": pd.DataFrame({"phi": stage2["phi"], "w2": stage2["w2"]}, index=criteria),
        "Objective_EWCS": pd.DataFrame({"p_raw": stage3_obj["p_raw"], "p_star": stage3_obj["p_star"]}, index=["w1", "w2"]),
        "OPA_Theta": opa_result["theta_df"],
        "OPA_Coefficients": opa_result["coeff_df"],
        "OPA_Weights": opa_result["weights_df"],
        "OPA_Ranked": opa_result["ranked_df"],
        "Final_Weights": final_weight_df,
        "Final_EWCS": pd.DataFrame({"p_raw": final_stage["p_raw"], "p_star": final_stage["p_star"]}, index=["objective", "subjective"]),
        "Fuzzy_Input": fuzzy_df,
        "Bonferroni_Normalized": norm_df,
        "SCoB": scob_df,
        "PCoB": pcob_df,
        "psi_a": psi_a_df,
        "psi_b": psi_b_df,
        "psi_c": psi_c_df,
        "Ranking": ranking_df,
        "CoCoSo_Meta": pd.DataFrame([cocoso_meta]),
    }

def export_results_to_excel(results: Dict[str, pd.DataFrame], filepath: str):
    with pd.ExcelWriter(filepath, engine="openpyxl") as writer:
        for name, df in results.items():
            if isinstance(df, pd.DataFrame):
                df.to_excel(writer, sheet_name=name[:31], index=True)


In [28]:
if __name__ == "__main__":

    D_df = pd.read_excel("scenario_summary.xlsx", index_col=0)

    # All criteria are COST criteria in your current dataset.
    types = ["C"] * 11

    # Calculate AL as 0.5 times the minimum value of each column in D_df
    AL = np.array([0.5 * D_df[col].min() for col in D_df.columns], dtype=float)

    # Calculate WL as 1.25 times the maximum value of each column in D_df
    WL = np.array([1.25 * D_df[col].max() for col in D_df.columns], dtype=float)

    # Calculate IT as 0.5 times the population standard deviation of each column in D_df
    IT = np.array([0.5 * D_df[col].std(ddof=0) for col in D_df.columns], dtype=float)

    # IMPORTANT: your linguistic matrix has 7 experts, so expert_weights must also contain 7 values.
    # If you do not have a separate weighting scheme for experts, use equal weights.
    expert_weights = [1/7] * 7

    linguistic_df = pd.DataFrame(
        {
            "E1":  ["ML", "AH", "AH", "AH", "ML", "VH", "VH", "AH", "VH", "AH", "AH"],
            "E2":  ["MH", "VH", "MH", "MH", "MH", "H",  "VH", "H",  "H",  "H",  "MH"],
            "E3":  ["VH", "VH", "L",  "L",  "L",  "E",  "AH", "AH", "E",  "E",  "E" ],
            "E4":  ["L",  "VH", "H",  "ML", "E",  "H",  "VH", "VH", "AH", "VH", "H" ],
            "E5":  ["AL", "AH", "L",  "L",  "L",  "VL", "H",  "ML", "L",  "H",  "H" ],
            "E6":  ["H",  "AH", "VH", "H",  "H",  "E",  "H",  "VH", "H",  "H",  "H" ],
            "E7":  ["H",  "VH", "ML", "E",  "ML", "H",  "VH", "H",  "H",  "ML", "H" ],
        },
        index=D_df.columns,
    )

    results = run_integrated_analysis(
        D_df=D_df,
        types=types,
        AL=AL,
        IT=IT,
        linguistic_df=linguistic_df,
        expert_weights=expert_weights,
        fuzzy_df=None,
        epsilon=0.10,
        phi1=1.0,
        phi2=1.0,
        pi_coef=0.5,
        WL=WL,
    )

    results_short = {
        "Decision Matrix": results["DecisionMatrix"],
        "Final_Weights": results["Final_Weights"],
        "Ranking": results["Ranking"],
    }

    # print("\n===== OBJECTIVE WEIGHTS =====")
    # print(results["ObjectiveSummary"].round(6).to_string(index=False))

    print("\n===== FINAL COMPOSITE CRITERION WEIGHTS =====")
    print(results["Final_Weights"].round(6).to_string(index=False))

    print("\n===== FINAL ALTERNATIVE RANKING =====")
    print(results["Ranking"][["Alternative", "Crisp", "Rank"]].round(6).to_string(index=False))

    export_results_to_excel(results, "integrated_mcdm_results.xlsx")
    print("\nExcel file saved as: integrated_mcdm_results.xlsx")

    export_results_to_excel(results_short, "integrated_mcdm_results_short.xlsx")
    print("\nExcel file saved as: integrated_mcdm_results_short.xlsx")




===== FINAL COMPOSITE CRITERION WEIGHTS =====
Criterion  Objective_weight  Subjective_weight  Final_weight  Rank
       C7          0.268622           0.147266      0.208435     1
       C2          0.065254           0.246290      0.155039     2
       C3          0.174518           0.050974      0.113246     3
       C8          0.062129           0.129345      0.095465     4
      C11          0.060561           0.112228      0.086185     5
      C10          0.066895           0.096250      0.081454     6
       C9          0.062158           0.080591      0.071299     7
       C6          0.053405           0.065095      0.059203     8
       C1          0.062142           0.036967      0.049656     9
       C4          0.062158           0.023944      0.043206    10
       C5          0.062158           0.011052      0.036812    11

===== FINAL ALTERNATIVE RANKING =====
Alternative      Crisp  Rank
         A1 295.384828     1
         A2 238.458859     2
         A3   0.687701 